# INDIA-METAGENE v6 — Fresh Rank 32, A100

## Why v6 (not continuing v5)
v5 tried to resume from a rank-8 checkpoint with rank-32 config — incompatible.
v6 starts **fresh from base METAGENE-1** with rank 32 from step 1.

## What's different
- Fresh LoRA rank 32 on all 4 attention layers (q/v/k/o)
- 200K sequences/epoch — 4x more data than v4
- 5 epochs — should reach ~4.60-4.65
- Anti-disconnect keep-alive
- Saves every 100 steps to HuggingFace
- Disconnect-proof: resumes from latest v6 checkpoint

## Expected results
```
Original METAGENE-1 : 4.8697
v4 best (rank 8)    : 4.8046
v6 target (rank 32) : ~4.60-4.65
Paper WW ref        : 1.2400
```

## Compute estimate
~1.5 hrs/epoch × 5 epochs = ~7.5 hrs = ~175 A100 units

## Before running
1. Runtime → Change runtime type → **A100 GPU**
2. Paste HF token in Cell 1
3. Runtime → Run all

In [ ]:
# Cell 1: Configuration
HF_TOKEN   = 'PASTE_YOUR_HF_TOKEN_HERE'
HF_REPO    = 'saurabhthakar3/gujarat-wastewater-shotgun'
MODEL_REPO = 'saurabhthakar3/india-metagene-1'

import os
os.environ['HF_TOKEN'] = HF_TOKEN
print('Token set.')

In [ ]:
# Cell 2: Anti-disconnect keep-alive
# Clicks reconnect every 3 minutes — browser tab must stay open
from IPython.display import display, Javascript
display(Javascript('''
function ClickConnect(){
    var buttons = document.querySelectorAll("colab-toolbar-button");
    for (var i = 0; i < buttons.length; i++) {
        if (buttons[i].id === "connect") {
            buttons[i].click();
            console.log("Keep-alive: reconnect clicked at " + new Date().toLocaleTimeString());
        }
    }
    // Dismiss any "session crashed" dialogs
    var dialogs = document.querySelectorAll('.dialogContent button');
    for (var d = 0; d < dialogs.length; d++) {
        if (dialogs[d].textContent.trim() === 'Run anyway' ||
            dialogs[d].textContent.trim() === 'Reconnect') {
            dialogs[d].click();
        }
    }
}
setInterval(ClickConnect, 180000);
console.log("Keep-alive started.");
'''))
print('Keep-alive active — runs every 3 min.')
print('IMPORTANT: Keep this browser tab open and active.')
print('Extra protection: chrome://discards → toggle off auto-discard for this tab.')

In [ ]:
# Cell 3: Install dependencies
import subprocess, sys
pkgs = [
    'transformers>=4.40', 'accelerate>=0.27', 'safetensors',
    'sentencepiece', 'huggingface_hub', 'scipy', 'scikit-learn',
    'seaborn', 'torchao>=0.16.0', 'peft>=0.10.0',
]
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade'] + pkgs, check=True)
import torchao, peft
print(f'torchao: {torchao.__version__} | peft: {peft.__version__}')
print('Done.')

In [ ]:
# Cell 4: GPU check
import torch
assert torch.cuda.is_available(), 'No GPU!'
gpu_name = torch.cuda.get_device_name(0)
vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'GPU : {gpu_name}')
print(f'VRAM: {vram_gb:.1f} GB')
assert vram_gb > 35, f'Need A100 (40GB), got {vram_gb:.1f}GB — change runtime'
BATCH_SIZE = 4
LORA_RANK  = 32
print(f'Batch size: {BATCH_SIZE} | LoRA rank: {LORA_RANK}')

In [ ]:
# Cell 5: Create/verify HuggingFace model repo
from huggingface_hub import HfApi
api = HfApi()
try:
    api.create_repo(repo_id=MODEL_REPO, repo_type='model',
                    private=True, token=HF_TOKEN)
    print(f'Created: {MODEL_REPO}')
except:
    print('Repo exists (OK).')

In [ ]:
# Cell 6: Download data + restore v6 checkpoint if exists
import shutil, json
from pathlib import Path
from huggingface_hub import snapshot_download, HfApi, hf_hub_download

DATA_ROOT      = Path('/content/gujarat_shotgun')
FASTA_DIR      = DATA_ROOT / 'phase3_fasta'
METADATA_CSV   = DATA_ROOT / 'sample_metadata_complete.csv'
CHECKPOINT_DIR = Path('/content/checkpoints')
RESULTS_DIR    = Path('/content/finetune_results')
MODEL_CACHE    = Path('/content/model_cache')

for d in [DATA_ROOT, CHECKPOINT_DIR, RESULTS_DIR, MODEL_CACHE]:
    d.mkdir(exist_ok=True)

# Download FASTA data
print('Downloading FASTA data...')
snapshot_download(
    repo_id=HF_REPO, repo_type='dataset',
    local_dir=str(DATA_ROOT), token=HF_TOKEN,
    ignore_patterns=['results/*','finetune_results*'],
)
print('Data downloaded.')

# Look for v6-specific checkpoints only
# v6 checkpoints are named: checkpoint_lora_v6_epoch*
print('\nChecking for v6 checkpoints...')
api         = HfApi()
START_EPOCH = 0
RESUME_CKPT = None

try:
    files = list(api.list_repo_files(
        repo_id=MODEL_REPO, repo_type='model', token=HF_TOKEN))
    # Only look for v6 checkpoints
    v6_ckpt_dirs   = set(f.split('/')[0] for f in files
                         if f.startswith('checkpoint_lora_v6_') and 'final' in f)
    if v6_ckpt_dirs:
        latest = sorted(v6_ckpt_dirs)[-1]
        print(f'Restoring v6 checkpoint: {latest}')
        snapshot_download(
            repo_id=MODEL_REPO, repo_type='model',
            local_dir=str(CHECKPOINT_DIR), token=HF_TOKEN,
            allow_patterns=[f'{latest}/*'],
        )
        RESUME_CKPT = CHECKPOINT_DIR / latest
        state_file  = RESUME_CKPT / 'training_state.json'
        if state_file.exists():
            state       = json.load(open(state_file))
            START_EPOCH = state['epoch']
            print(f'Resuming v6 from epoch {START_EPOCH}')
    else:
        print('No v6 checkpoints found — starting fresh from base model.')
except Exception as e:
    print(f'Checkpoint check failed: {e} — starting fresh.')

# Restore v6 training history
history_path = RESULTS_DIR / 'training_history_v6.json'
try:
    files = list(api.list_repo_files(
        repo_id=MODEL_REPO, repo_type='model', token=HF_TOKEN))
    if 'training_history_v6.json' in files:
        hf_hub_download(
            repo_id=MODEL_REPO, repo_type='model',
            filename='training_history_v6.json',
            local_dir=str(RESULTS_DIR), token=HF_TOKEN
        )
        print('v6 history restored.')
except: pass

fastas = sorted(FASTA_DIR.glob('*.fasta.gz'))
print(f'\nFASTA files : {len(fastas)}')
print(f'Start epoch : {START_EPOCH}')
print(f'Resume ckpt : {RESUME_CKPT}')
assert len(fastas) > 0

In [ ]:
# Cell 7: Stratified split (same seed as all previous versions)
import pandas as pd, random
random.seed(42)

meta_raw = pd.read_csv(METADATA_CSV)
fasta_df = pd.DataFrame({
    'fasta_path': fastas,
    'fasta_id'  : [f.stem.replace('.fasta','') for f in fastas],
    'meta_key'  : [f.stem.replace('.fasta','').replace('_R1','') for f in fastas],
})
fasta_df = fasta_df.merge(meta_raw, left_on='meta_key', right_on='sample_id', how='left')

CITIES      = sorted(fasta_df['city'].dropna().unique())
SEASONS     = ['Summer','Monsoon','PreWinter','Winter']
CITY_COLORS = {'Ahmedabad':'#1f77b4','Gandhinagar':'#ff7f0e',
               'Rajkot':'#2ca02c','Vadodara':'#d62728'}
SEASON_ORDER = ['Summer','Monsoon','PreWinter','Winter']

val_ids = []
for city in CITIES:
    for season in SEASONS:
        subset = fasta_df[
            (fasta_df['city']==city) & (fasta_df['season']==season)
        ]['fasta_id'].tolist()
        if subset:
            val_ids.append(random.choice(subset))

train_fastas = [f for f in fastas if f.stem.replace('.fasta','') not in val_ids]
val_fastas   = [f for f in fastas if f.stem.replace('.fasta','') in val_ids]
print(f'Train: {len(train_fastas)} | Val: {len(val_fastas)} (1 per city × season)')

In [ ]:
# Cell 8: Load METAGENE-1 with FRESH rank 32 LoRA
# KEY: Only resumes from v6 checkpoints — never loads v4/v5 incompatible weights
import torch, time
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model, TaskType, PeftModel

device   = torch.device('cuda')
MODEL_ID = 'metagene-ai/METAGENE-1'

print(f'Loading base model {MODEL_ID} in float16...')
t0 = time.time()

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID, cache_dir=str(MODEL_CACHE))

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    cache_dir=str(MODEL_CACHE),
    torch_dtype=torch.float16,
    device_map='cuda',
)

if RESUME_CKPT and RESUME_CKPT.exists():
    # Resume from a v6 checkpoint (rank 32, compatible)
    print(f'Resuming from v6 checkpoint: {RESUME_CKPT}')
    model = PeftModel.from_pretrained(
        model, str(RESUME_CKPT), is_trainable=True)
    print('v6 checkpoint loaded.')
else:
    # Fresh start — rank 32, all 4 attention layers
    print(f'Applying FRESH LoRA rank={LORA_RANK} on q/v/k/o layers...')
    lora_config = LoraConfig(
        task_type=TaskType.CAUSAL_LM,
        r=LORA_RANK,
        lora_alpha=LORA_RANK * 2,
        lora_dropout=0.05,
        target_modules=['q_proj','v_proj','k_proj','o_proj'],
        bias='none',
    )
    model = get_peft_model(model, lora_config)
    print('Fresh LoRA applied.')

model.print_trainable_parameters()
vram  = torch.cuda.memory_allocated()/1e9
total = torch.cuda.get_device_properties(0).total_memory/1e9
print(f'\nLoaded in {time.time()-t0:.0f}s')
print(f'VRAM: {vram:.1f} / {total:.1f} GB  ({total-vram:.1f} GB free)')

In [ ]:
# Cell 9: Float32 fix — CRITICAL for learning
import torch

n_fixed = 0
for name, param in model.named_parameters():
    if param.requires_grad:
        param.data = param.data.float()
        n_fixed += 1

trainable_dtypes = set(p.dtype for p in model.parameters() if p.requires_grad)
frozen_dtypes    = set(p.dtype for p in model.parameters() if not p.requires_grad)
print(f'Fixed {n_fixed} LoRA tensors → float32')
print(f'Trainable: {trainable_dtypes}')  # must be float32
print(f'Frozen   : {frozen_dtypes}')     # float16
print(f'VRAM     : {torch.cuda.memory_allocated()/1e9:.1f} GB')
assert torch.float32 in trainable_dtypes, 'ERROR: LoRA not float32!'
print('Float32 fix applied — ready to train.')

In [ ]:
# Cell 10: Helpers + config
import gzip, random, json
import numpy as np
from huggingface_hub import HfApi
from torch.amp import autocast, GradScaler

SEED               = 42
MAX_SEQ_LEN        = 512
GRAD_ACCUM         = 8
LR                 = 3e-5
MAX_EPOCHS         = 5
N_SEQS_PER_EPOCH   = 200_000
SAVE_EVERY_N_STEPS = 100

random.seed(SEED); np.random.seed(SEED)


def sample_sequences(fasta_list, n_total, seed=SEED):
    rng        = random.Random(seed)
    result     = []
    n_per_file = max(1, n_total // len(fasta_list))
    shuffled   = list(fasta_list); rng.shuffle(shuffled)
    for fasta in shuffled:
        opener = gzip.open if str(fasta).endswith('.gz') else open
        seqs   = []
        with opener(fasta, 'rt') as f:
            seq = []
            for line in f:
                line = line.rstrip()
                if line.startswith('>'):
                    if seq: seqs.append(''.join(seq))
                    seq = []
                    if len(seqs) >= n_per_file: break
                else: seq.append(line)
        result.extend(seqs[:n_per_file])
        if len(result) >= n_total: break
    rng.shuffle(result)
    return result[:n_total]


def make_batches(seqs, bs=BATCH_SIZE):
    for i in range(0, len(seqs), bs):
        yield seqs[i:i+bs]


@torch.no_grad()
def evaluate_loss(fasta_list, n_seqs=1000):
    model.eval()
    seqs   = sample_sequences(fasta_list, n_seqs)
    losses = []
    for batch in make_batches(seqs, bs=8):
        inputs = tokenizer(
            batch, return_tensors='pt', padding=True,
            truncation=True, max_length=MAX_SEQ_LEN,
            add_special_tokens=False
        ).to(device)
        with autocast('cuda', dtype=torch.float16):
            outputs = model(**inputs, labels=inputs['input_ids'])
        sl = outputs.logits[...,:-1,:].contiguous().float()
        tl = inputs['input_ids'][...,1:].contiguous()
        sm = inputs['attention_mask'][...,1:].contiguous().float()
        loss_fct = torch.nn.CrossEntropyLoss(reduction='none')
        tok_loss = loss_fct(
            sl.view(-1,sl.size(-1)), tl.view(-1)).view(tl.size())
        norm_loss = (tok_loss*sm).sum(1)/sm.sum(1).clamp(min=1)
        losses.extend(norm_loss.cpu().float().numpy().tolist())
        del outputs, inputs
        torch.cuda.empty_cache()
    model.train()
    return float(np.mean(losses)), float(np.std(losses))


def upload_file(local_path, hf_path, silent=False):
    for attempt in range(3):
        try:
            HfApi().upload_file(
                path_or_fileobj=str(local_path),
                path_in_repo=hf_path,
                repo_id=MODEL_REPO,
                repo_type='model',
                token=HF_TOKEN
            )
            if not silent: print(f'  ✓ {hf_path}')
            return True
        except Exception as e:
            if attempt < 2:
                import time; time.sleep(5)
            else:
                if not silent: print(f'  Upload failed: {e}')
    return False


def save_checkpoint(epoch, step, train_loss, val_loss):
    # v6 checkpoints use v6_ prefix to distinguish from v4/v5
    ckpt_name = f'lora_v6_epoch{epoch}_step{step}'
    ckpt_path = CHECKPOINT_DIR / ckpt_name
    ckpt_path.mkdir(exist_ok=True)
    model.save_pretrained(str(ckpt_path))
    tokenizer.save_pretrained(str(ckpt_path))
    json.dump(
        {'epoch':epoch, 'step':step, 'train_loss':train_loss,
         'val_loss':val_loss, 'lora_rank':LORA_RANK,
         'version':'v6'},
        open(ckpt_path/'training_state.json','w'), indent=2
    )
    print(f'  Uploading {ckpt_name}...', end=' ')
    try:
        HfApi().upload_folder(
            folder_path=str(ckpt_path),
            repo_id=MODEL_REPO, repo_type='model',
            path_in_repo=f'checkpoint_{ckpt_name}',
            token=HF_TOKEN,
        )
        print('✓')
    except Exception as e:
        print(f'Upload failed: {e}')


steps_per_epoch = N_SEQS_PER_EPOCH // BATCH_SIZE // GRAD_ACCUM
est_hrs         = steps_per_epoch * GRAD_ACCUM * BATCH_SIZE / 17 / 3600

print('Helpers ready.')
print(f'LoRA rank        : {LORA_RANK} (fresh from base model)')
print(f'Sequences/epoch  : {N_SEQS_PER_EPOCH:,}')
print(f'Effective batch  : {BATCH_SIZE} × {GRAD_ACCUM} = {BATCH_SIZE*GRAD_ACCUM}')
print(f'Steps/epoch      : {steps_per_epoch:,}')
print(f'Total epochs     : {MAX_EPOCHS}')
print(f'Est. hrs/epoch   : {est_hrs:.1f} on A100')
print(f'Est. total time  : {est_hrs*MAX_EPOCHS:.1f} hrs')
print(f'Save every       : {SAVE_EVERY_N_STEPS} steps')

In [ ]:
# Cell 11: Baseline
import json, time
from huggingface_hub import hf_hub_download

baseline_path = RESULTS_DIR / 'baseline.json'

if baseline_path.exists():
    baseline          = json.load(open(baseline_path))
    baseline_val_mean = baseline['val_mean']
    print(f'Baseline from file: {baseline_val_mean:.4f}')
else:
    try:
        hf_hub_download(
            repo_id=MODEL_REPO, repo_type='model',
            filename='baseline.json',
            local_dir=str(RESULTS_DIR), token=HF_TOKEN
        )
        baseline          = json.load(open(baseline_path))
        baseline_val_mean = baseline['val_mean']
        print(f'Baseline from HF: {baseline_val_mean:.4f}')
    except:
        print('Computing baseline (takes ~2 min)...')
        baseline_val_mean, baseline_val_std = evaluate_loss(val_fastas, n_seqs=2000)
        baseline = {
            'val_mean'   : baseline_val_mean,
            'val_std'    : baseline_val_std,
            'paper_ww'   : 1.24,
            'paper_human': 5.22,
        }
        json.dump(baseline, open(baseline_path,'w'), indent=2)
        upload_file(baseline_path, 'baseline.json')
        print(f'Baseline: {baseline_val_mean:.4f}')

print(f'\nOriginal METAGENE-1 : {baseline_val_mean:.4f}')
print(f'v4 best (rank 8)    : 4.8046')
print(f'v6 target (rank 32) : ~4.60-4.65')
print(f'Paper WW reference  : 1.2400')

In [ ]:
# Cell 12: Fine-tuning loop
# v6: fresh rank 32, 200K seqs/epoch, saves every 100 steps
import torch, time, json, random
import numpy as np
from torch.optim import AdamW
from torch.amp import autocast, GradScaler
from transformers import get_linear_schedule_with_warmup
from huggingface_hub import HfApi

# Load or init v6 history
history_path = RESULTS_DIR / 'training_history_v6.json'
if history_path.exists():
    history = json.load(open(history_path))
    print(f'v6 history: {len(history["epoch_val_loss"])} epochs done')
    if history['epoch_val_loss']:
        print(f'Best val loss so far: {min(history["epoch_val_loss"]):.4f}')
else:
    history = {
        'epoch_train_loss': [],
        'epoch_val_loss'  : [],
        'step_losses'     : [],
        'epochs_done'     : [],
        'lora_rank'       : LORA_RANK,
        'n_seqs_per_epoch': N_SEQS_PER_EPOCH,
        'version'         : 'v6',
        'v4_reference'    : 4.8046,
        'baseline'        : baseline_val_mean,
    }

# Optimizer
trainable_params = [p for p in model.parameters() if p.requires_grad]
optimizer = AdamW(trainable_params, lr=LR, weight_decay=0.01)
scaler    = GradScaler('cuda')

steps_per_epoch = N_SEQS_PER_EPOCH // BATCH_SIZE // GRAD_ACCUM
remaining       = MAX_EPOCHS - START_EPOCH
total_steps     = steps_per_epoch * remaining
warmup_steps    = min(200, total_steps // 10)
scheduler       = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps
)

print(f'v6 Training — FRESH rank {LORA_RANK} from base METAGENE-1')
print(f'Starting epoch : {START_EPOCH+1} / {MAX_EPOCHS}')
print(f'Sequences/epoch: {N_SEQS_PER_EPOCH:,}')
print(f'Steps/epoch    : {steps_per_epoch:,}')
print(f'LR             : {LR}')
print(f'Save every     : {SAVE_EVERY_N_STEPS} steps')
print('─'*70)

for epoch in range(START_EPOCH, MAX_EPOCHS):
    print(f'\n=== EPOCH {epoch+1}/{MAX_EPOCHS} (v6, rank {LORA_RANK}) ===')
    t_epoch = time.time()

    print('Sampling sequences...')
    epoch_seqs = sample_sequences(
        train_fastas, N_SEQS_PER_EPOCH, seed=SEED+epoch)
    print(f'Sampled {len(epoch_seqs):,}')

    epoch_losses = []
    accum_loss   = 0.0
    global_step  = 0
    optimizer.zero_grad()
    model.train()

    for step, batch in enumerate(make_batches(epoch_seqs)):
        inputs = tokenizer(
            batch, return_tensors='pt', padding=True,
            truncation=True, max_length=MAX_SEQ_LEN,
            add_special_tokens=False
        ).to(device)

        with autocast('cuda', dtype=torch.float16):
            outputs = model(**inputs, labels=inputs['input_ids'])
            loss    = outputs.loss / GRAD_ACCUM

        scaler.scale(loss).backward()
        accum_loss += loss.item()
        del outputs, inputs
        torch.cuda.empty_cache()

        if (step + 1) % GRAD_ACCUM == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(trainable_params, 1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            optimizer.zero_grad()

            step_loss = accum_loss
            epoch_losses.append(step_loss)
            history['step_losses'].append(step_loss)
            accum_loss  = 0.0
            global_step += 1

            if global_step % 50 == 0:
                elapsed = time.time() - t_epoch
                rate    = (step+1)*BATCH_SIZE / elapsed
                pct     = (step+1) / len(epoch_seqs) * BATCH_SIZE * 100
                eta_min = (len(epoch_seqs)/BATCH_SIZE-step-1)/max(rate,1)*BATCH_SIZE/60
                print(f'  Step {global_step:>5} | '
                      f'loss={step_loss:.4f} | '
                      f'{pct:.1f}% | '
                      f'{rate:.0f} seq/s | '
                      f'ETA {eta_min:.0f} min')

            if global_step % SAVE_EVERY_N_STEPS == 0:
                print(f'\n  --- Checkpoint step {global_step} ---')
                val_mean, val_std = evaluate_loss(val_fastas, n_seqs=500)
                print(f'  Val loss   : {val_mean:.4f} ± {val_std:.4f}')
                print(f'  vs baseline: {baseline_val_mean-val_mean:+.4f}')
                print(f'  vs v4      : {4.8046-val_mean:+.4f}')
                save_checkpoint(
                    epoch+1, global_step,
                    float(np.mean(epoch_losses[-50:])),
                    val_mean
                )
                json.dump(history, open(history_path,'w'), indent=2)
                upload_file(history_path,
                            'training_history_v6.json', silent=True)
                model.train()

    # End of epoch
    epoch_train_loss = float(np.mean(epoch_losses))
    elapsed_min = (time.time()-t_epoch)/60
    print(f'\nEpoch {epoch+1} done | {elapsed_min:.0f} min')
    print(f'  Train loss : {epoch_train_loss:.4f}')

    val_mean, val_std = evaluate_loss(val_fastas, n_seqs=2000)
    print(f'  Val loss   : {val_mean:.4f} ± {val_std:.4f}')
    print(f'  vs baseline: {baseline_val_mean-val_mean:+.4f} '
          f'({(baseline_val_mean-val_mean)/baseline_val_mean*100:.2f}%)')
    print(f'  vs v4      : {4.8046-val_mean:+.4f}  '
          f'← improvement over previous best')

    history['epoch_train_loss'].append(epoch_train_loss)
    history['epoch_val_loss'].append(val_mean)
    history['epochs_done'].append(epoch+1)
    json.dump(history, open(history_path,'w'), indent=2)

    save_checkpoint(epoch+1, 'final', epoch_train_loss, val_mean)
    upload_file(history_path, 'training_history_v6.json', silent=True)

    START_EPOCH = epoch + 1

print('\n' + '='*65)
print('v6 TRAINING COMPLETE')
print(f'Final val loss : {history["epoch_val_loss"][-1]:.4f}')
print(f'v4 reference   : 4.8046')
print(f'Improvement    : {4.8046-history["epoch_val_loss"][-1]:+.4f}')
print(f'vs baseline    : {baseline_val_mean-history["epoch_val_loss"][-1]:+.4f}')
print('='*65)

In [ ]:
# Cell 13: Training curves
import matplotlib.pyplot as plt
import numpy as np, json

history  = json.load(open(RESULTS_DIR/'training_history_v6.json'))
baseline = json.load(open(RESULTS_DIR/'baseline.json'))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
sl = history['step_losses']
if len(sl) > 10:
    w  = min(100, len(sl)//10)
    sm = np.convolve(sl, np.ones(w)/w, mode='valid')
    ax.plot(sl, alpha=0.15, color='steelblue')
    ax.plot(range(w-1,len(sl)), sm, color='steelblue',
            lw=2, label='v6 smoothed')
ax.axhline(baseline['val_mean'], color='red', ls='--', lw=1.5,
           label=f'Baseline ({baseline["val_mean"]:.3f})')
ax.axhline(4.8046, color='grey', ls=':', lw=1.5,
           label='v4 final (4.8046)')
ax.axhline(baseline['paper_ww'], color='green', ls='--', lw=1.5,
           label=f'Paper WW ({baseline["paper_ww"]})')
ax.set_xlabel('Step'); ax.set_ylabel('CE Loss')
ax.set_title(f'v6 Training Loss (rank {LORA_RANK})', fontweight='bold')
ax.legend(fontsize=8)

ax = axes[1]
# v4 reference points
v4_losses = [baseline['val_mean'],4.8525,4.8343,4.8311,4.8142,
             4.8113,4.8074,4.806,4.8052,4.8052,4.8046]
ax.plot(range(len(v4_losses)), v4_losses, 'o--',
        color='grey', lw=1.5, ms=5, alpha=0.7, label='v4 (rank 8)')

if history.get('epoch_val_loss'):
    v6_epochs = [0] + history['epochs_done']
    v6_losses = [baseline['val_mean']] + history['epoch_val_loss']
    ax.plot(v6_epochs, v6_losses, 'o-',
            color='steelblue', lw=2, ms=8, label=f'v6 (rank {LORA_RANK})')

ax.axhline(baseline['val_mean'], color='red', ls='--', lw=1.5,
           label=f'Baseline ({baseline["val_mean"]:.3f})')
ax.axhline(baseline['paper_ww'], color='green', ls='--', lw=1.5,
           label=f'Paper WW ({baseline["paper_ww"]})')
ax.axhline(baseline['paper_human'], color='orange', ls='--', lw=1.5,
           label=f'Human genome ({baseline["paper_human"]})')
ax.set_xlabel('Epoch'); ax.set_ylabel('Val CE Loss')
ax.set_title('Val Loss: v4 vs v6', fontweight='bold')
ax.legend(fontsize=8)

plt.suptitle(f'INDIA-METAGENE v6: rank {LORA_RANK}, 200K seq/epoch',
             fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig(RESULTS_DIR/'training_curve_v6.pdf', dpi=150, bbox_inches='tight')
plt.savefig(RESULTS_DIR/'training_curve_v6.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'v6 val losses: {[round(v,4) for v in history["epoch_val_loss"]]}')

In [ ]:
# Cell 14: Per-sample evaluation
import torch, json, numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

baseline = json.load(open(RESULTS_DIR/'baseline.json'))
V4_FINAL = 4.8046

print('Per-sample val loss (v6)...')
per_sample = []
for fasta in val_fastas:
    sid      = fasta.stem.replace('.fasta','')
    meta_key = sid.replace('_R1','')
    row      = meta_raw[meta_raw['sample_id']==meta_key]
    city     = row['city'].iloc[0]   if len(row) else 'Unknown'
    season   = row['season'].iloc[0] if len(row) else 'Unknown'
    mean_loss, _ = evaluate_loss([fasta], n_seqs=500)
    per_sample.append({
        'sample_id'  : sid, 'city': city, 'season': season,
        'v6_loss'    : mean_loss,
        'baseline'   : baseline['val_mean'],
        'v4_loss'    : V4_FINAL,
        'vs_baseline': baseline['val_mean'] - mean_loss,
        'vs_v4'      : V4_FINAL - mean_loss,
    })
    print(f'  {sid:<15} {city:<12} {season:<10} '
          f'base={baseline["val_mean"]:.3f} '
          f'v4={V4_FINAL:.3f} '
          f'v6={mean_loss:.3f} '
          f'(Δv4→v6={V4_FINAL-mean_loss:+.3f})')

results_df = pd.DataFrame(per_sample)
results_df.to_csv(RESULTS_DIR/'finetuned_val_results_v6.csv', index=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
ax = axes[0]
x  = np.arange(len(results_df)); w = 0.35
ax.bar(x-w/2, results_df['baseline'], w,
       alpha=0.4, color='red', label='Baseline')
ax.bar(x+w/2, results_df['v6_loss'], w,
       color=[CITY_COLORS.get(c,'grey') for c in results_df['city']],
       alpha=0.9, label='v6 (rank 32)')
ax.axhline(1.24, color='green', ls='--', lw=1.5, label='Paper WW')
ax.axhline(V4_FINAL, color='grey', ls=':', lw=1.5, label=f'v4 ({V4_FINAL})')
ax.set_xticks(x)
ax.set_xticklabels(results_df['sample_id'], rotation=90, fontsize=7)
ax.set_ylabel('Mean CE Loss')
ax.set_title('Baseline vs v6 per sample', fontweight='bold')
ax.legend(fontsize=7)

ax = axes[1]
s_ord = [s for s in SEASON_ORDER if s in results_df['season'].values]
sns.boxplot(data=results_df, x='season', y='v6_loss',
            order=s_ord, hue='season', palette='Set1',
            ax=ax, legend=False)
ax.axhline(baseline['val_mean'], color='red', ls='--', lw=1.5,
           label=f'Baseline ({baseline["val_mean"]:.3f})')
ax.axhline(V4_FINAL, color='grey', ls=':', lw=1.5,
           label=f'v4 ({V4_FINAL})')
ax.axhline(1.24, color='green', ls='--', lw=1.5, label='Paper WW')
ax.set_title('v6 Loss by Season', fontweight='bold')
ax.set_ylabel('Mean CE Loss'); ax.legend(fontsize=8)

plt.suptitle(f'INDIA-METAGENE v6 Results (rank {LORA_RANK})',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(RESULTS_DIR/'finetuning_results_v6.pdf', dpi=150, bbox_inches='tight')
plt.savefig(RESULTS_DIR/'finetuning_results_v6.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n=== v6 FINAL SUMMARY ===')
print(f'Original METAGENE-1 : {baseline["val_mean"]:.4f}')
print(f'v4 (rank 8, 10ep)   : {V4_FINAL:.4f}')
print(f'v6 (rank 32, 5ep)   : '
      f'{results_df["v6_loss"].mean():.4f} '
      f'± {results_df["v6_loss"].std():.4f}')
print(f'v6 vs v4            : '
      f'{V4_FINAL-results_df["v6_loss"].mean():+.4f}')
print(f'v6 vs baseline      : '
      f'{baseline["val_mean"]-results_df["v6_loss"].mean():+.4f}')
print(f'Paper WW reference  : 1.2400')

In [ ]:
# Cell 15: Save all results to HuggingFace
from huggingface_hub import HfApi
api = HfApi()
print('Saving results...')
skip = {'.cache'}
for f in RESULTS_DIR.glob('*'):
    if f.name in skip or f.name.startswith('.'): continue
    try:
        api.upload_file(
            path_or_fileobj=str(f),
            path_in_repo=f'finetune_results_v6/{f.name}',
            repo_id=HF_REPO,
            repo_type='dataset',
            token=HF_TOKEN
        )
        print(f'  ✓ {f.name}')
    except Exception as e:
        print(f'  ✗ {f.name}: {e}')
print('All results saved.')